# Notebook 05 — S3-Trigger CI/CD Pipeline (Simplified: Jupyter Monitoring Loop)

**team03 — Ong Hui Lin (Student 2, MLOps & Deployment)**

This notebook builds a second, *triggered* version of the SageMaker Pipeline
from Notebook 03. Instead of being started manually with a hardcoded dataset
path, this pipeline accepts the training data location as a **runtime
parameter** (`InputDataUrl`) and is started automatically whenever a new CSV
file lands in a watched S3 prefix — a minimal, notebook-based stand-in for a
production CI/CD retraining trigger.

Steps: **PreprocessData → TrainModel → AUCQualityGate → RegisterModel**
(RegisterModel only runs if the quality gate passes) — the exact same
four-step shape as Notebook 03's manual pipeline, so the two are easy to
explain side by side; the only real difference is that the training data
location is now a runtime parameter instead of a hardcoded path.


> **Adaptation note.** The team's tutor-materials folder does not contain a
> literal `s3-data-trigger-pipeline.html` reference notebook — only a
> conceptual walkthrough in `mlops_pipeline_updated_v2.pdf` ("File 4:
> s3-data-trigger-pipeline.html", pages 19–25). That walkthrough describes two
> versions of the trigger:
>
> 1. **Full production chain** — S3 upload → EventBridge rule → Lambda
>    function → `SageMaker Pipeline.start()`.
> 2. **Simplified classroom version** — a Jupyter cell polls a watched S3
>    prefix directly and calls `start_pipeline_execution()` itself, with no
>    EventBridge/Lambda/CloudWatch setup required.
>
> This notebook builds version 2, which the tutor recommends as the "main
> lab" path. GitHub Actions / the full EventBridge→Lambda chain is explicitly
> marked as an **advanced extension** in the tutor's material and is
> intentionally skipped here — consistent with Notebook 03's own
> `## 6. GitHub Actions CI/CD [SKIP THIS]` section.
>
> `preprocess.py`, `train.py`, and `inference.py` are reused **unchanged**
> from Notebook 03. Simplified 1 Aug 2026: this notebook originally added a
> separate `evaluate.py` Processing step (with a `PropertyFile` + `JsonGet`
> quality gate) to independently re-score the model, decoupled from training
> log parsing. That extra step wasn't required by the tutor's material and
> made the pipeline harder to explain alongside Notebook 03's simpler
> version, so it's been removed — the quality gate here now reads
> `step_train.properties.FinalMetricDataList["test_auc_roc"]`, **exactly**
> the same pattern Notebook 03 already uses. One less script, one less
> Processing job, and the two pipelines are now easy to describe as "the
> same four steps, just with a parameterized input."
>
> Both pipelines (Notebook 03's manual pipeline and this notebook's triggered
> pipeline) register approved models to the **same** Model Registry group
> (`{TEAM_ID}-CryptoScamDetector`), so every run — manual or triggered — shows
> up as a new version in one place, which is what the Progress Check rubric's
> "model registry available for versioning" tier is checking for.


## 0. Configuration

In [1]:
import boto3
import sagemaker
import json
import os
import time
import glob
from pathlib import Path
from datetime import datetime

session = sagemaker.Session()
role    = sagemaker.get_execution_role()
region  = boto3.Session().region_name

BUCKET  = "nyp-26s1-iti113"

TEAM_ID = "team03"
STUDENT_ID = "s301"

COURSE = "ITI113"
SEMESTER = "26S1"
PROJECT_NAME = "crypto-scam-detector"

PREFIX  = f"iti113/{TEAM_ID}/data/{PROJECT_NAME}"

PROCESSING_INSTANCE_TYPE = "ml.m5.large"
TRAINING_INSTANCE_TYPE   = "ml.m5.large"

# Same manual pipeline + raw dataset as Notebook 03 (used as the default
# InputDataUrl when nothing has triggered this pipeline yet).
MANUAL_PIPELINE_NAME = f"iti113-{TEAM_ID}-crypto-scam-detector"
RAW_DATA_URI          = f"s3://{BUCKET}/{PREFIX}/raw/crypto_scam_dataset.csv"

# This notebook's triggered pipeline -- separate pipeline object, same
# Model Registry group, so manual and triggered runs both version into one
# registry.
TRIGGERED_PIPELINE_NAME = f"iti113-{TEAM_ID}-crypto-scam-detector-triggered"
MODEL_PACKAGE_GROUP     = f"{TEAM_ID}-CryptoScamDetector"

# ADAPTED: matches Notebook 02/03's ROC-AUC threshold, not the tutor's example value.
QUALITY_GATE_AUC = 0.85

PIPELINE_ROOT = f"s3://{BUCKET}/{PREFIX}/pipeline-triggered"

SCRIPTS_S3_PREFIX = f"{PREFIX}/pipeline_src"
SCRIPTS_S3_URI    = f"s3://{BUCKET}/{SCRIPTS_S3_PREFIX}"
LOCAL_PIPELINE_SRC = "pipeline_src"

# ----------------------------
# S3-trigger watch folder (stand-in for the production EventBridge rule)
# ----------------------------
TRIGGER_PREFIX          = f"{PREFIX}/trigger"
TRIGGER_INCOMING_PREFIX = f"{TRIGGER_PREFIX}/incoming"
TRIGGER_INCOMING_URI    = f"s3://{BUCKET}/{TRIGGER_INCOMING_PREFIX}"
SEEN_STATE_FILE         = Path(f"trigger_seen_keys_{TEAM_ID}.json")

print(f"Manual pipeline (Notebook 03)  : {MANUAL_PIPELINE_NAME}")
print(f"Triggered pipeline (this NB)   : {TRIGGERED_PIPELINE_NAME}")
print(f"Model Registry group           : {MODEL_PACKAGE_GROUP}")
print(f"Bucket                         : {BUCKET}")
print(f"Team prefix                    : {PREFIX}")
print(f"Region                         : {region}")
print(f"SageMaker role                 : {role}")
print(f"Pipeline source S3 URI         : {SCRIPTS_S3_URI}")
print(f"Trigger watch folder           : {TRIGGER_INCOMING_URI}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Manual pipeline (Notebook 03)  : iti113-team03-crypto-scam-detector
Triggered pipeline (this NB)   : iti113-team03-crypto-scam-detector-triggered
Model Registry group           : team03-CryptoScamDetector
Bucket                         : nyp-26s1-iti113
Team prefix                    : iti113/team03/data/crypto-scam-detector
Region                         : ap-southeast-1
SageMaker role                 : arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team03
Pipeline source S3 URI         : s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src
Trigger watch folder           : s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/trigger/incoming


## 1. Pipeline Scripts

`preprocess.py`, `train.py`, and `inference.py` are byte-for-byte the same
scripts written in Notebook 03 (retraining must use identical feature
engineering and model code, or a "triggered" model would not be comparable
to the manual one). This notebook writes its own copies so it can run
standalone without depending on Notebook 03 having been executed first in
the same kernel/session.


In [2]:
os.makedirs('src', exist_ok=True)
print('src/ directory ready')

src/ directory ready


In [3]:
%%writefile src/preprocess.py
"""SageMaker Processing Job -- replicates Notebooks 01/02 preprocessing for text data.

Cleans the raw scam-message text, builds the engineered indicator features
(urgency, contact/link, structural characteristics), fits TF-IDF on the
training split only, and saves everything the training job and the deployed
endpoint need to stay in sync: TF-IDF features (sparse .npz), engineered
features and labels (CSV), and a preprocessor bundle (joblib) containing the
fitted vectorizer.
"""
import os
import re
import argparse
import glob

import pandas as pd
import numpy as np
import joblib
import scipy.sparse as sp

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

parser = argparse.ArgumentParser()
parser.add_argument('--test-size',    type=float, default=0.20)
parser.add_argument('--random-state', type=int,   default=42)
args = parser.parse_args()

# BUGFIX (Progress Check debugging, 1 Aug 2026): this used to be hardcoded to
# 'crypto_scam_dataset.csv', which only matched the manual pipeline's fixed raw
# file. Notebook 05's whole point is that InputDataUrl -- and therefore the
# downloaded filename -- changes with whatever CSV triggered the run, so the
# hardcoded name caused a FileNotFoundError ("AlgorithmError, exit code: 1")
# the first time a differently-named file (synthetic_batch_manual-test.csv)
# triggered this script. Discover the file instead of assuming its name.
input_dir = '/opt/ml/processing/input'
csv_candidates = sorted(glob.glob(os.path.join(input_dir, '*.csv')))
if not csv_candidates:
    raise FileNotFoundError(
        f'No CSV file found in {input_dir}. Expected the file that triggered '
        'this pipeline run (or crypto_scam_dataset.csv for a manual run).'
    )
input_path = csv_candidates[0]
print(f'Using input file: {input_path}')
output_dir = '/opt/ml/processing/output'
os.makedirs(output_dir, exist_ok=True)

SCAM_LABEL = "scam"

# ---------------------------------------------------------------------------
# Text cleaning (mirrors utils/preprocessing.py's clean_text())
# ---------------------------------------------------------------------------
URL_PATTERN = re.compile(r"https?://[^\s]+|www\.[^\s]+")

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = URL_PATTERN.sub(" <url> ", text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# ---------------------------------------------------------------------------
# Engineered indicator features (mirrors utils/indicators.py + utils/preprocessing.py)
# ---------------------------------------------------------------------------
URGENT_KEYWORDS = [
    "urgent", "immediately", "act now", "act fast", "hurry", "limited time",
    "today only", "expires today", "offer ends soon", "last chance",
    "don't miss out", "within 24 hours", "within 1 hour", "respond now",
    "claim now", "limited slots", "before it's too late", "time-sensitive",
]
GUARANTEED_RETURN_KEYWORDS = [
    "guaranteed return", "guaranteed returns", "guaranteed profit",
    "risk-free", "risk free", "100% profit", "double your money",
    "high returns", "\u7a33\u8d5a\u4e0d\u8d54",
]
PAYMENT_KEYWORDS = [
    "deposit", "transfer funds", "send payment", "pay now", "top up",
    "bitcoin", "btc", "ethereum", "eth", "usdt", "wallet address",
]
OFF_PLATFORM_KEYWORDS = [
    "telegram", "whatsapp", "discord", "wechat", "private chat",
    "dm me", "direct message",
]
CREDENTIAL_KEYWORDS = [
    "seed phrase", "private key", "wallet password", "recovery phrase",
    "otp", "verification code", "security code",
]

WALLET_PATTERN = re.compile(r"\b(?:0x[a-fA-F0-9]{40}|[13][a-km-zA-HJ-NP-Z1-9]{25,34})\b")
EMAIL_PATTERN = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")
PHONE_PATTERN = re.compile(r"(?:\+?\d[\d\s-]{7,}\d)")
COUNTDOWN_PATTERN = re.compile(r"\b\d+\s*(?:hour|hours|hr|hrs|minute|minutes|min|mins|day|days)\b", re.IGNORECASE)

def find_keyword_matches(message, keywords):
    message_lower = message.lower()
    return [k for k in keywords if k.lower() in message_lower]

def extract_engineered_features(text):
    if not isinstance(text, str):
        text = ""

    urgency_keyword_hits = len(find_keyword_matches(text, URGENT_KEYWORDS))
    guaranteed_return_hits = len(find_keyword_matches(text, GUARANTEED_RETURN_KEYWORDS))
    countdown_hits = len(COUNTDOWN_PATTERN.findall(text))
    exclamation_count = text.count("!")
    urgency_score = urgency_keyword_hits + countdown_hits + min(exclamation_count, 3)

    wallet_matches = WALLET_PATTERN.findall(text)
    url_matches = URL_PATTERN.findall(text)
    email_matches = EMAIL_PATTERN.findall(text)
    phone_matches = PHONE_PATTERN.findall(text)
    payment_hits = len(find_keyword_matches(text, PAYMENT_KEYWORDS))
    off_platform_hits = len(find_keyword_matches(text, OFF_PLATFORM_KEYWORDS))
    credential_hits = len(find_keyword_matches(text, CREDENTIAL_KEYWORDS))

    letters = [c for c in text if c.isalpha()]
    capital_ratio = sum(1 for c in letters if c.isupper()) / len(letters) if letters else 0.0
    digit_count = sum(1 for c in text if c.isdigit())

    return {
        "urgency_keyword_count": urgency_keyword_hits,
        "guaranteed_return_keyword_count": guaranteed_return_hits,
        "countdown_phrase_count": countdown_hits,
        "exclamation_count": exclamation_count,
        "urgency_score": urgency_score,
        "has_wallet_address": int(bool(wallet_matches)),
        "wallet_address_count": len(wallet_matches),
        "has_url": int(bool(url_matches)),
        "url_count": len(url_matches),
        "has_email": int(bool(email_matches)),
        "has_phone_number": int(bool(phone_matches)),
        "payment_keyword_count": payment_hits,
        "off_platform_keyword_count": off_platform_hits,
        "credential_keyword_count": credential_hits,
        "message_length": len(text),
        "capital_letter_ratio": round(capital_ratio, 4),
        "has_numeric_content": int(digit_count > 0),
        "digit_count": digit_count,
    }

ENGINEERED_COLS = [
    "urgency_keyword_count", "guaranteed_return_keyword_count", "countdown_phrase_count",
    "exclamation_count", "urgency_score", "has_wallet_address", "wallet_address_count",
    "has_url", "url_count", "has_email", "has_phone_number", "payment_keyword_count",
    "off_platform_keyword_count", "credential_keyword_count", "message_length",
    "capital_letter_ratio", "has_numeric_content", "digit_count",
]

# ---------------------------------------------------------------------------
# Load, clean, engineer, split
# ---------------------------------------------------------------------------
df = pd.read_csv(input_path)
expected_columns = {"id", "platform", "text", "label"}
missing = expected_columns - set(df.columns)
if missing:
    raise ValueError(f"Dataset is missing expected columns: {missing}")

df["clean_text"] = df["text"].apply(clean_text)

engineered = pd.DataFrame([extract_engineered_features(t) for t in df["text"]], index=df.index)
df = pd.concat([df, engineered], axis=1)

train_df, test_df = train_test_split(
    df,
    test_size=args.test_size,
    random_state=args.random_state,
    stratify=df["label"],
)

print(f"Train: {train_df.shape[0]} rows | Test: {test_df.shape[0]} rows")

# ---------------------------------------------------------------------------
# TF-IDF -- fit on TRAIN text only, never on test
# ---------------------------------------------------------------------------
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)
vectorizer.fit(train_df["clean_text"])

X_train_tfidf = vectorizer.transform(train_df["clean_text"])
X_test_tfidf = vectorizer.transform(test_df["clean_text"])

y_train = (train_df["label"] == SCAM_LABEL).astype(int)
y_test = (test_df["label"] == SCAM_LABEL).astype(int)

print(f"TF-IDF vocabulary size: {len(vectorizer.get_feature_names_out())}")

# ---------------------------------------------------------------------------
# Save outputs -- sparse TF-IDF as .npz (dense CSV would be hundreds of MB
# for a 5000-column matrix), everything else as CSV.
# ---------------------------------------------------------------------------
sp.save_npz(f"{output_dir}/train_tfidf.npz", X_train_tfidf)
sp.save_npz(f"{output_dir}/test_tfidf.npz", X_test_tfidf)

train_df[ENGINEERED_COLS].to_csv(f"{output_dir}/train_engineered.csv", index=False)
test_df[ENGINEERED_COLS].to_csv(f"{output_dir}/test_engineered.csv", index=False)

y_train.to_csv(f"{output_dir}/train_labels.csv", index=False, header=True)
y_test.to_csv(f"{output_dir}/test_labels.csv", index=False, header=True)

preprocessor = {
    "vectorizer": vectorizer,
    "engineered_cols": ENGINEERED_COLS,
    "feature_columns": list(vectorizer.get_feature_names_out()) + ENGINEERED_COLS,
}
joblib.dump(preprocessor, f"{output_dir}/preprocessor.joblib")

print("Preprocessing complete. Saved TF-IDF (.npz), engineered features, labels, and preprocessor.joblib.")
print(f"Final feature count: {len(preprocessor['feature_columns'])}")

Overwriting src/preprocess.py


In [4]:
%%writefile src/train.py
"""
SageMaker Training Job
-----------------------
Trains a Random Forest classifier on TF-IDF + engineered features and saves
a deployment bundle.

Hyperparameters default to the winning configuration from Notebook 02's
hyperparameter tuning (rf_candidate_05: n_estimators=200, max_depth=8,
min_samples_leaf=2, test AUC-ROC from that run), overridable as SageMaker
Pipeline parameters.

The deployment bundle contains the trained model AND the fitted TF-IDF
vectorizer (from the Processing step's preprocessor.joblib), so the endpoint
can accept raw message text and reproduce training-time feature engineering.

MLflow logging is intentionally performed outside the SageMaker training
container by the notebook after a successful pipeline execution.
"""
import os
import argparse
import pickle
import joblib
import pandas as pd
import scipy.sparse as sp

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score,
)

parser = argparse.ArgumentParser()
parser.add_argument('--n-estimators', type=int, default=200)
parser.add_argument('--max-depth', type=int, default=8)
parser.add_argument('--min-samples-leaf', type=int, default=2)
parser.add_argument('--random-state', type=int, default=42)

parser.add_argument('--team-id', type=str, default=os.environ.get('TEAM_ID', 'unknown-team'))
parser.add_argument('--student-id', type=str, default=os.environ.get('STUDENT_ID', 's000'))
parser.add_argument('--semester', type=str, default=os.environ.get('SEMESTER', '26S1'))
parser.add_argument('--run-name', type=str, default='sagemaker_pipeline_run')

parser.add_argument(
    '--model-dir',
    type=str,
    default=os.environ.get('SM_MODEL_DIR', '/opt/ml/model')
)
parser.add_argument(
    '--train',
    type=str,
    default=os.environ.get('SM_CHANNEL_TRAIN', '/opt/ml/input/data/train')
)
parser.add_argument(
    '--test',
    type=str,
    default=os.environ.get('SM_CHANNEL_TEST', '/opt/ml/input/data/test')
)
args = parser.parse_args()

os.makedirs(args.model_dir, exist_ok=True)

print('=== SageMaker Training Environment ===')
print(f'Train channel: {args.train}')
print(f'Test channel: {args.test}')
print(f'Model directory: {args.model_dir}')

X_train_tfidf = sp.load_npz(os.path.join(args.train, 'train_tfidf.npz'))
X_test_tfidf = sp.load_npz(os.path.join(args.test, 'test_tfidf.npz'))

train_engineered = pd.read_csv(os.path.join(args.train, 'train_engineered.csv'))
test_engineered = pd.read_csv(os.path.join(args.test, 'test_engineered.csv'))

y_train = pd.read_csv(os.path.join(args.train, 'train_labels.csv')).squeeze('columns')
y_test = pd.read_csv(os.path.join(args.test, 'test_labels.csv')).squeeze('columns')

preprocessor_path = os.path.join(args.train, 'preprocessor.joblib')
if not os.path.exists(preprocessor_path):
    raise FileNotFoundError(
        f'preprocessor.joblib was not found at {preprocessor_path}. '
        'Rerun the ProcessingStep with the updated preprocess.py.'
    )

preprocessor = joblib.load(preprocessor_path)
engineered_cols = preprocessor['engineered_cols']

X_train = sp.hstack([X_train_tfidf, train_engineered[engineered_cols].values]).tocsr()
X_test = sp.hstack([X_test_tfidf, test_engineered[engineered_cols].values]).tocsr()

print(f'Train: {X_train.shape[0]} rows, {X_train.shape[1]} features')
print(f'Test : {X_test.shape[0]}  rows')

if len(pd.Series(y_train).unique()) < 2:
    raise ValueError('Training labels contain fewer than two classes.')

model = RandomForestClassifier(
    n_estimators=args.n_estimators,
    max_depth=args.max_depth,
    min_samples_leaf=args.min_samples_leaf,
    class_weight='balanced',
    random_state=args.random_state,
    n_jobs=-1,
)
model.fit(X_train, y_train)

all_metrics = {}
for split, X, y in [('train', X_train, y_train), ('test', X_test, y_test)]:
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]

    all_metrics.update({
        f'{split}_accuracy': round(accuracy_score(y, predictions), 4),
        f'{split}_f1': round(f1_score(y, predictions, zero_division=0), 4),
        f'{split}_precision': round(
            precision_score(y, predictions, zero_division=0), 4
        ),
        f'{split}_recall': round(
            recall_score(y, predictions, zero_division=0), 4
        ),
    })

    if len(pd.Series(y).unique()) >= 2:
        all_metrics[f'{split}_auc_roc'] = round(
            roc_auc_score(y, probabilities), 4
        )
    else:
        all_metrics[f'{split}_auc_roc'] = None
        print(f'Warning: {split} split has only one class; AUC-ROC unavailable.')

print('=== Metrics ===')
for metric_name, metric_value in all_metrics.items():
    print(f'{metric_name}: {metric_value}')

model_bundle = {
    'model': model,
    'preprocessor': preprocessor,
    'engineered_cols': engineered_cols,
    'input_format': 'raw_text_json',
    'description': (
        'Deployment bundle containing trained model and fitted TF-IDF vectorizer. '
        'Endpoint accepts raw message text as JSON: {"text": "..."}'
    ),
}

joblib.dump(model_bundle, os.path.join(args.model_dir, 'model.joblib'))

with open(os.path.join(args.model_dir, 'model.pkl'), 'wb') as f:
    pickle.dump(model_bundle, f)

print(f"Model bundle saved: {os.path.join(args.model_dir, 'model.joblib')}")
print(f"Legacy bundle saved: {os.path.join(args.model_dir, 'model.pkl')}")

if all_metrics['test_auc_roc'] is None:
    raise ValueError('Test AUC-ROC is unavailable; cannot evaluate the quality gate.')

print(f"Test AUC-ROC: {all_metrics['test_auc_roc']}")
print(f"test_accuracy: {all_metrics['test_accuracy']}")
print(f"test_f1: {all_metrics['test_f1']}")

Overwriting src/train.py


In [5]:
%%writefile src/inference.py
"""SageMaker inference handler for the deployed crypto-scam-detector endpoint.

Accepts JSON input with raw message text:

{"text": "Deposit 500 USDT today and receive guaranteed returns..."}

or a list of such records, or {"instances": [...]}. Applies the same text
cleaning, engineered-feature extraction, and TF-IDF transform saved in
model.joblib before calling the trained model -- the same logic used in
Notebooks 01/02 and utils/preprocessing.py, so serving matches training.
"""
import os
import re
import json
import pickle
import joblib
import pandas as pd
import scipy.sparse as sp

URL_PATTERN = re.compile(r"https?://[^\s]+|www\.[^\s]+")

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = URL_PATTERN.sub(" <url> ", text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

URGENT_KEYWORDS = [
    "urgent", "immediately", "act now", "act fast", "hurry", "limited time",
    "today only", "expires today", "offer ends soon", "last chance",
    "don't miss out", "within 24 hours", "within 1 hour", "respond now",
    "claim now", "limited slots", "before it's too late", "time-sensitive",
]
GUARANTEED_RETURN_KEYWORDS = [
    "guaranteed return", "guaranteed returns", "guaranteed profit",
    "risk-free", "risk free", "100% profit", "double your money",
    "high returns", "\u7a33\u8d5a\u4e0d\u8d54",
]
PAYMENT_KEYWORDS = [
    "deposit", "transfer funds", "send payment", "pay now", "top up",
    "bitcoin", "btc", "ethereum", "eth", "usdt", "wallet address",
]
OFF_PLATFORM_KEYWORDS = [
    "telegram", "whatsapp", "discord", "wechat", "private chat",
    "dm me", "direct message",
]
CREDENTIAL_KEYWORDS = [
    "seed phrase", "private key", "wallet password", "recovery phrase",
    "otp", "verification code", "security code",
]

WALLET_PATTERN = re.compile(r"\b(?:0x[a-fA-F0-9]{40}|[13][a-km-zA-HJ-NP-Z1-9]{25,34})\b")
EMAIL_PATTERN = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")
PHONE_PATTERN = re.compile(r"(?:\+?\d[\d\s-]{7,}\d)")
COUNTDOWN_PATTERN = re.compile(r"\b\d+\s*(?:hour|hours|hr|hrs|minute|minutes|min|mins|day|days)\b", re.IGNORECASE)

def find_keyword_matches(message, keywords):
    message_lower = message.lower()
    return [k for k in keywords if k.lower() in message_lower]

def extract_engineered_features(text):
    if not isinstance(text, str):
        text = ""

    urgency_keyword_hits = len(find_keyword_matches(text, URGENT_KEYWORDS))
    guaranteed_return_hits = len(find_keyword_matches(text, GUARANTEED_RETURN_KEYWORDS))
    countdown_hits = len(COUNTDOWN_PATTERN.findall(text))
    exclamation_count = text.count("!")
    urgency_score = urgency_keyword_hits + countdown_hits + min(exclamation_count, 3)

    wallet_matches = WALLET_PATTERN.findall(text)
    url_matches = URL_PATTERN.findall(text)
    email_matches = EMAIL_PATTERN.findall(text)
    phone_matches = PHONE_PATTERN.findall(text)
    payment_hits = len(find_keyword_matches(text, PAYMENT_KEYWORDS))
    off_platform_hits = len(find_keyword_matches(text, OFF_PLATFORM_KEYWORDS))
    credential_hits = len(find_keyword_matches(text, CREDENTIAL_KEYWORDS))

    letters = [c for c in text if c.isalpha()]
    capital_ratio = sum(1 for c in letters if c.isupper()) / len(letters) if letters else 0.0
    digit_count = sum(1 for c in text if c.isdigit())

    return {
        "urgency_keyword_count": urgency_keyword_hits,
        "guaranteed_return_keyword_count": guaranteed_return_hits,
        "countdown_phrase_count": countdown_hits,
        "exclamation_count": exclamation_count,
        "urgency_score": urgency_score,
        "has_wallet_address": int(bool(wallet_matches)),
        "wallet_address_count": len(wallet_matches),
        "has_url": int(bool(url_matches)),
        "url_count": len(url_matches),
        "has_email": int(bool(email_matches)),
        "has_phone_number": int(bool(phone_matches)),
        "payment_keyword_count": payment_hits,
        "off_platform_keyword_count": off_platform_hits,
        "credential_keyword_count": credential_hits,
        "message_length": len(text),
        "capital_letter_ratio": round(capital_ratio, 4),
        "has_numeric_content": int(digit_count > 0),
        "digit_count": digit_count,
    }


def model_fn(model_dir):
    """Load the model bundle from the SageMaker model directory."""
    joblib_path = os.path.join(model_dir, 'model.joblib')
    pkl_path = os.path.join(model_dir, 'model.pkl')

    if os.path.exists(joblib_path):
        bundle = joblib.load(joblib_path)
    elif os.path.exists(pkl_path):
        with open(pkl_path, 'rb') as f:
            bundle = pickle.load(f)
    else:
        raise FileNotFoundError('Neither model.joblib nor model.pkl was found.')

    return bundle


def input_fn(body, content_type='application/json'):
    """Parse JSON request body into a DataFrame of raw message-text records."""
    if content_type != 'application/json':
        raise ValueError(f'Unsupported content type: {content_type}')

    payload = json.loads(body)

    if isinstance(payload, dict):
        if 'instances' in payload:
            payload = payload['instances']
        else:
            payload = [payload]

    if not isinstance(payload, list):
        raise ValueError('JSON input must be a dictionary, a list of dictionaries, or {"instances": [...]}')

    return pd.DataFrame(payload)


def predict_fn(data, bundle):
    """Clean text, extract engineered features, TF-IDF transform, then predict."""
    if 'text' not in data.columns:
        raise ValueError('Input must include a "text" field with the raw message.')

    model = bundle['model']
    preprocessor = bundle['preprocessor']
    vectorizer = preprocessor['vectorizer']
    engineered_cols = preprocessor['engineered_cols']

    clean = data['text'].apply(clean_text)
    engineered = pd.DataFrame(
        [extract_engineered_features(t) for t in data['text']], index=data.index
    )

    X_tfidf = vectorizer.transform(clean)
    X = sp.hstack([X_tfidf, engineered[engineered_cols].values]).tocsr()

    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]
    return predictions, probabilities


def output_fn(prediction, accept='application/json'):
    preds, probas = prediction
    response = [
        {
            'prediction': int(p),
            'label': 'Scam' if int(p) == 1 else 'Legitimate',
            'probability': round(float(b), 4),
        }
        for p, b in zip(preds, probas)
    ]
    return json.dumps(response), accept


Overwriting src/inference.py


In [6]:
print("Scripts written:")
for fn in ["preprocess.py", "train.py", "inference.py"]:
    size = os.path.getsize(f"src/{fn}")
    print(f"  src/{fn}  ({size} bytes)")


Scripts written:
  src/preprocess.py  (9111 bytes)
  src/train.py  (5701 bytes)
  src/inference.py  (6573 bytes)


## 1A. Upload pipeline source files to S3

Uploads `preprocess.py`, `train.py`, and `inference.py` to the team's
pipeline source prefix (the same prefix Notebook 03 uses), then downloads
them into a local `pipeline_src/` folder that the pipeline step definitions
below reference.


In [7]:
s3_client = boto3.client("s3")

FILES_TO_UPLOAD = ["preprocess.py", "train.py", "inference.py"]

for filename in FILES_TO_UPLOAD:
    local_path = Path("src") / filename
    s3_key = f"{SCRIPTS_S3_PREFIX}/{filename}"

    if not local_path.exists():
        raise FileNotFoundError(f"Missing local source file: {local_path}")

    s3_client.upload_file(str(local_path), BUCKET, s3_key)
    print(f"Uploaded {local_path} -> s3://{BUCKET}/{s3_key}")

print("Pipeline source files uploaded to:", SCRIPTS_S3_URI)


Uploaded src/preprocess.py -> s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src/preprocess.py
Uploaded src/train.py -> s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src/train.py
Uploaded src/inference.py -> s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src/inference.py
Pipeline source files uploaded to: s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src


In [8]:
# from pathlib import Path

# SEEN_STATE_FILE = Path("trigger_seen_keys_team03.json")
# if SEEN_STATE_FILE.exists():
#     SEEN_STATE_FILE.unlink()
#     print("✓ Deleted trigger state file")
# else:
#     print("State file doesn't exist (OK, will create on next baseline)")

In [9]:
import shutil

local_src = Path(LOCAL_PIPELINE_SRC)

if local_src.exists():
    shutil.rmtree(local_src)

local_src.mkdir(parents=True, exist_ok=True)

for filename in FILES_TO_UPLOAD:
    s3_key = f"{SCRIPTS_S3_PREFIX}/{filename}"
    local_path = local_src / filename

    s3_client.download_file(BUCKET, s3_key, str(local_path))
    print(f"Downloaded s3://{BUCKET}/{s3_key} -> {local_path}")

print("Downloaded files:")
for p in sorted(local_src.iterdir()):
    print("-", p)

Downloaded s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src/preprocess.py -> pipeline_src/preprocess.py
Downloaded s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src/train.py -> pipeline_src/train.py


Downloaded s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src/inference.py -> pipeline_src/inference.py
Downloaded files:
- pipeline_src/inference.py
- pipeline_src/preprocess.py
- pipeline_src/train.py


## 2. Define the Triggered SageMaker Pipeline

Same four building blocks as Notebook 03 (Processing → Training → Condition
→ Register), with one change: `ProcessingStep`'s input source is now a
**pipeline parameter** (`InputDataUrl`) instead of a hardcoded S3 path, so
whichever CSV triggered this run gets trained on. Everything else — the
quality gate, the registration step — is identical to Notebook 03.


In [10]:
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.parameters import ParameterFloat, ParameterInteger, ParameterString
from sagemaker.workflow.model_step import ModelStep
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.model import Model
from sagemaker.workflow.pipeline_context import PipelineSession

# ADAPTED: default_bucket/default_bucket_prefix keep every SDK-managed upload
# (estimator source code, pipeline definition, etc.) inside the team's own S3
# prefix. Without this, the SDK sometimes uploads to a default location
# outside nyp-26s1-iti113/iti113/team03/... and IAM correctly rejects it with
# S3UploadFailedError: AccessDenied.
pipeline_session = PipelineSession(default_bucket=BUCKET, default_bucket_prefix=PREFIX)

# Pipeline parameters -- can be overridden at execution time.
# InputDataUrl is the runtime parameter the S3 trigger sets: whichever CSV
# landed in the watch folder becomes this run's training data.
p_input_data = ParameterString(name='InputDataUrl', default_value=RAW_DATA_URI)
p_n_est      = ParameterInteger(name='NEstimators',    default_value=200)  # matches rf_candidate_05
p_depth      = ParameterInteger(name='MaxDepth',       default_value=8)   # matches rf_candidate_05
p_samples    = ParameterInteger(name='MinSamplesLeaf', default_value=2)   # matches rf_candidate_05
p_gate       = ParameterFloat(  name='QualityGateAUC', default_value=QUALITY_GATE_AUC)

print('Pipeline parameters defined.')


Pipeline parameters defined.


In [11]:
# Step 1: ProcessingStep -- trains on whichever file triggered this run
processor = SKLearnProcessor(
    framework_version='1.2-1', instance_type=PROCESSING_INSTANCE_TYPE,
    instance_count=1, role=role, sagemaker_session=pipeline_session,
    base_job_name=f'iti113-{TEAM_ID}-{STUDENT_ID}-process-triggered')

step_process = ProcessingStep(
    name='PreprocessData',
    processor=processor,
    inputs=[ProcessingInput(source=p_input_data,
                            destination='/opt/ml/processing/input')],
    outputs=[ProcessingOutput(output_name='processed',
                              source='/opt/ml/processing/output',
                              destination=f'{PIPELINE_ROOT}/processed')],
    code=f'{LOCAL_PIPELINE_SRC}/preprocess.py',
    job_arguments=['--test-size','0.2','--random-state','42']
)
print('Step 1 (ProcessingStep) defined.')

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


/opt/conda/lib/python3.12/site-packages/sagemaker/processing.py:138: SageMakerV2DeprecationWarning: SKLearnProcessor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `DataProcessor` (`from sagemaker.mlops.processing import DataProcessor`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Step 1 (ProcessingStep) defined.


In [12]:
# Step 2: TrainingStep -- identical to Notebook 03's
estimator = SKLearn(
    entry_point="train.py",
    source_dir=LOCAL_PIPELINE_SRC,
    framework_version="1.2-1",
    instance_type=TRAINING_INSTANCE_TYPE,
    instance_count=1,
    role=role,
    base_job_name=f"iti113-{TEAM_ID}-{STUDENT_ID}-train-triggered",
    sagemaker_session=pipeline_session,
    hyperparameters={
        "n-estimators": p_n_est,
        "max-depth": p_depth,
        "min-samples-leaf": p_samples,
        "random-state": 42,
        "team-id": TEAM_ID,
        "student-id": STUDENT_ID,
        "semester": SEMESTER,
        "run-name": "triggered_pipeline_run",
    },
    environment={
        "TEAM_ID": TEAM_ID,
        "STUDENT_ID": STUDENT_ID,
        "SEMESTER": SEMESTER,
    },
    metric_definitions=[
        {"Name": "test_auc_roc", "Regex": "Test AUC-ROC: ([0-9\\.]+)"},
        {"Name": "test_accuracy", "Regex": "test_accuracy: ([0-9\\.]+)"},
        {"Name": "test_f1", "Regex": "test_f1: ([0-9\\.]+)"},
    ],
    tags=[
        {"Key": "Course", "Value": "ITI113"},
        {"Key": "Semester", "Value": SEMESTER},
        {"Key": "Team", "Value": TEAM_ID},
        {"Key": "Student", "Value": STUDENT_ID},
    ],
)

processed_uri = step_process.properties.ProcessingOutputConfig.Outputs["processed"].S3Output.S3Uri

step_train = TrainingStep(
    name="TrainModel",
    estimator=estimator,
    inputs={
        # No content_type -- Processing output is a mix of .npz (TF-IDF) and
        # .csv (engineered features/labels); train.py reads each file directly.
        "train": sagemaker.inputs.TrainingInput(s3_data=processed_uri),
        "test": sagemaker.inputs.TrainingInput(s3_data=processed_uri),
    },
)

print("Step 2 (TrainingStep) defined.")

/opt/conda/lib/python3.12/site-packages/sagemaker/estimator.py:588: SageMakerV2DeprecationWarning: SKLearn is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Step 2 (TrainingStep) defined.


In [13]:
# Step 3: ModelStep -- register in the SAME Model Registry group Notebook 03
# uses, so manual and triggered runs both version into one place.
model = Model(
    image_uri=estimator.training_image_uri(region),
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    sagemaker_session=pipeline_session,
    role=role,
    entry_point='inference.py',
    source_dir=LOCAL_PIPELINE_SRC
)
step_register = ModelStep(
    name='RegisterModel',
    step_args=model.register(
        content_types=['application/json'],
        response_types=['application/json'],
        inference_instances=['ml.m5.large'],
        transform_instances=['ml.m5.large'],
        model_package_group_name=MODEL_PACKAGE_GROUP,
        approval_status='PendingManualApproval',
    )
)
print('Step 3 (ModelStep) defined.')


/opt/conda/lib/python3.12/site-packages/sagemaker/model.py:347: SageMakerV2DeprecationWarning: Model is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


Step 3 (ModelStep) defined.


In [14]:
# Step 4: ConditionStep -- gate on SageMaker-captured test AUC, using the
# EXACT same pattern as Notebook 03's manual pipeline: read the metric
# straight from the TrainingStep's captured CloudWatch metric
# (estimator.metric_definitions already regex-captures "Test AUC-ROC: 0.xxxx"
# from train.py's stdout). Simplified 1 Aug 2026 -- this used to read from a
# separate evaluate.py step's evaluation.json via PropertyFile/JsonGet; that
# extra Processing step wasn't needed and made this pipeline harder to
# explain side-by-side with Notebook 03, so it's gone. One fewer script, one
# fewer Processing job, identical quality-gate logic in both pipelines.
condition = ConditionGreaterThanOrEqualTo(
    left=step_train.properties.FinalMetricDataList["test_auc_roc"].Value,
    right=p_gate
)

step_condition = ConditionStep(
    name="AUCQualityGate",
    conditions=[condition],
    if_steps=[step_register],
    else_steps=[]
)
print("Step 4 (ConditionStep) defined.")


Step 4 (ConditionStep) defined.


In [15]:
# Assemble and upsert the triggered pipeline
pipeline_triggered = Pipeline(
    name=TRIGGERED_PIPELINE_NAME,
    parameters=[p_input_data, p_n_est, p_depth, p_samples, p_gate],
    steps=[step_process, step_train, step_condition],
    sagemaker_session=pipeline_session
)
pipeline_triggered.upsert(role_arn=role)
print(f'Pipeline "{TRIGGERED_PIPELINE_NAME}" upserted.')
print('View in SageMaker Studio: left sidebar -> Pipelines')


/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline.py:119: SageMakerV2DeprecationWarning: Pipeline is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `Pipeline` (`from sagemaker.mlops.pipeline import Pipeline`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Pipeline "iti113-team03-crypto-scam-detector-triggered" upserted.
View in SageMaker Studio: left sidebar -> Pipelines


## 3. Prove the Trigger Works End-to-End

This section is the notebook stand-in for "Lambda function logic": it checks
the watched S3 prefix for files not seen before, and for each new one, starts
`pipeline_triggered` with `InputDataUrl` set to that file's S3 URI.

First, take a baseline snapshot of what is already in the watch folder, so
pre-existing files are not treated as "new" the first time this runs.

In [16]:
def list_trigger_keys():
    s3 = boto3.client('s3')
    keys = set()
    paginator = s3.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=BUCKET, Prefix=f'{TRIGGER_INCOMING_PREFIX}/'):
        for obj in page.get('Contents', []):
            if obj['Key'].endswith('.csv'):
                keys.add(obj['Key'])
    return keys

def load_seen_keys():
    if SEEN_STATE_FILE.exists():
        return set(json.loads(SEEN_STATE_FILE.read_text()))
    return set()

def save_seen_keys(keys):
    SEEN_STATE_FILE.write_text(json.dumps(sorted(keys)))

# Baseline: anything already in the folder counts as "already seen".
seen_keys = load_seen_keys() | list_trigger_keys()
save_seen_keys(seen_keys)
print(f'Baseline established: {len(seen_keys)} existing file(s) in {TRIGGER_INCOMING_URI}/')
print('Any file uploaded after this point will be treated as a new trigger.')

Baseline established: 6 existing file(s) in s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/trigger/incoming/
Any file uploaded after this point will be treated as a new trigger.


### 3a. Simulate a new batch of scraped messages arriving

Uploads a small synthetic CSV to the watch folder. **This is not meant to be
a realistic retraining batch** — like the tutor's own trigger-test example,
it exists only to prove the S3-trigger mechanism actually fires, using the
same schema (`id`, `platform`, `text`, `label`) as the real dataset.

In [17]:
import csv
import io
from datetime import datetime

synthetic_rows = [
    (90001, "Telegram", "URGENT: Your wallet qualifies for a guaranteed 100% profit airdrop! Deposit 500 USDT within 1 hour to claim now, don't miss out!", "scam"),
    (90002, "X", "Guaranteed returns, risk-free! Double your money in 24 hours, DM me your wallet address to get started today only.", "scam"),
    (90003, "Discord", "Send your seed phrase to our support bot to verify your account and unlock your staking rewards immediately.", "scam"),
    (90004, "Email", "Act now! Limited slots left for our risk-free ETH investment pool, 100% profit guaranteed, reply with your BTC wallet.", "scam"),
    (90005, "WhatsApp", "Last chance, offer ends soon: transfer funds to this wallet address for a guaranteed high-return crypto giveaway.", "scam"),
    (90006, "Reddit", "Anyone else been staking ETH through a validator lately? Curious what APY people are actually seeing right now.", "legitimate"),
    (90007, "X", "Just finished reading the latest L2 rollup research paper, really interesting tradeoffs on data availability.", "legitimate"),
    (90008, "Discord", "Our dev team pushed a testnet update this morning, changelog is in the pinned message if anyone wants to review it.", "legitimate"),
    (90009, "Telegram", "Market's been choppy this week, mostly sitting in stables until there's a clearer trend to trade.", "legitimate"),
    (90010, "Email", "Reminder: the community call for the protocol governance vote is scheduled for next Tuesday at 3pm UTC.", "legitimate"),
]

buf = io.StringIO()
writer = csv.writer(buf)
writer.writerow(["id", "platform", "text", "label"])
writer.writerows(synthetic_rows)

# Use current timestamp to make filename unique
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
synthetic_key = f"{TRIGGER_INCOMING_PREFIX}/synthetic_batch_{timestamp}.csv"

s3_client = boto3.client("s3")
s3_client.put_object(Bucket=BUCKET, Key=synthetic_key, Body=buf.getvalue().encode("utf-8"))

synthetic_uri = f"s3://{BUCKET}/{synthetic_key}"
print(f"✓ Uploaded new batch: {synthetic_uri}")

✓ Uploaded new batch: s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/trigger/incoming/synthetic_batch_20260801-092115.csv


### 3b. Check the watch folder and trigger a run

This is the function a Lambda would run in the production version. Here it
runs as an ordinary notebook cell: list the folder, diff against what has
been seen before, and start the triggered pipeline for each new file.

In [18]:
def check_trigger_folder_once(auto_start=True):
    """Check the watch folder for files not seen before. For each new file,
    optionally start pipeline_triggered with InputDataUrl set to that file.
    Returns the list of newly detected S3 URIs."""
    current_keys = list_trigger_keys()
    seen = load_seen_keys()
    new_keys = sorted(current_keys - seen)

    if not new_keys:
        print("No new files detected.")
        return []

    new_uris = []
    for key in new_keys:
        uri = f"s3://{BUCKET}/{key}"
        print(f"New file detected: {uri}")
        new_uris.append(uri)

        if auto_start:
            execution = pipeline_triggered.start(parameters={
                'InputDataUrl': uri,
                'NEstimators': 200, 'MaxDepth': 8, 'MinSamplesLeaf': 2,
                'QualityGateAUC': QUALITY_GATE_AUC,
            })
            print(f"  -> Started execution: {execution.arn}")

    save_seen_keys(seen | current_keys)
    return new_uris

detected = check_trigger_folder_once(auto_start=True)
print(f"\n{len(detected)} new file(s) triggered a pipeline execution this check.")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


New file detected: s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/trigger/incoming/synthetic_batch_20260801-092115.csv


  -> Started execution: arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/iti113-team03-crypto-scam-detector-triggered/execution/9992vwikqcjp

1 new file(s) triggered a pipeline execution this check.


If a new execution was started above, check its status. A full run takes
roughly 10-15 minutes end to end (Preprocess -> Train -> Evaluate ->
quality gate -> Register); this cell only reports the *current* status
without blocking, so re-run it later to see progress.

In [39]:
sm = boto3.client('sagemaker', region_name=region)

result = pipeline_triggered.list_executions()

# ADAPTED: some SageMaker SDK versions return the raw
# list_pipeline_executions() response dict (with a "PipelineExecutionSummaries"
# key) instead of a plain list of execution summaries -- handle both shapes.
if isinstance(result, dict):
    summaries = result.get("PipelineExecutionSummaries", [])
else:
    summaries = result

if summaries:
    latest = summaries[0]
    exec_arn = latest['PipelineExecutionArn']
    print(f"Latest execution : {exec_arn}")
    print(f"Status           : {latest['PipelineExecutionStatus']}")

    # Step-by-step breakdown -- shows which step failed and why, instead of
    # just the overall pipeline status.
    steps_resp = sm.list_pipeline_execution_steps(PipelineExecutionArn=exec_arn)
    print("\nStep-by-step status:")
    for step in steps_resp.get('PipelineExecutionSteps', []):
        print(f"  {step.get('StepName')}: {step.get('StepStatus')}")
        failure_reason = step.get('FailureReason')
        if failure_reason:
            print(f"    FailureReason: {failure_reason}")
else:
    print("No executions found yet for this pipeline.")

Latest execution : arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/iti113-team03-crypto-scam-detector-triggered/execution/9992vwikqcjp
Status           : Succeeded

Step-by-step status:
  RegisterModel-RegisterModel: Succeeded
  RegisterModel-RepackModel-0: Succeeded
  AUCQualityGate: Succeeded
  TrainModel: Succeeded
  PreprocessData: Succeeded


In [38]:
sm = boto3.client('sagemaker', region_name=region)

# Get the latest execution
exec_result = pipeline_triggered.list_executions()
if isinstance(exec_result, dict):
    summaries = exec_result.get("PipelineExecutionSummaries", [])
else:
    summaries = exec_result

if summaries:
    latest = summaries[0]
    exec_arn = latest['PipelineExecutionArn']
    print(f"Latest execution: {exec_arn}")
    print(f"Status: {latest['PipelineExecutionStatus']}\n")

    # Check if RegisterModel step completed
    steps_resp = sm.list_pipeline_execution_steps(PipelineExecutionArn=exec_arn)
    register_step = [s for s in steps_resp.get('PipelineExecutionSteps', []) 
                     if s['StepName'] == 'RegisterModel']
    
    if register_step:
        register_status = register_step[0]['StepStatus']
        print(f"RegisterModel step: {register_status}")
        
        if register_status == 'Completed':
            print("✓ Model was registered!\n")
            
            # List all model versions in the registry
            models = sm.list_model_packages(
                ModelPackageGroupName=MODEL_PACKAGE_GROUP,
                SortBy='CreationTime',
                SortOrder='Descending'
            )
            
            print(f"Model Package Group: {MODEL_PACKAGE_GROUP}")
            print(f"Total versions: {len(models['ModelPackageSummaryList'])}\n")
            
            if models['ModelPackageSummaryList']:
                latest_model = models['ModelPackageSummaryList'][0]
                print(f"Latest model version:")
                print(f"  ARN: {latest_model['ModelPackageArn']}")
                print(f"  Status: {latest_model['ModelPackageStatus']}")
                print(f"  Created: {latest_model['CreationTime']}")
        else:
            print(f"✗ RegisterModel step did not complete: {register_status}")
    else:
        print("✗ RegisterModel step not found in pipeline")
else:
    print("No executions found")

Latest execution: arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/iti113-team03-crypto-scam-detector-triggered/execution/9992vwikqcjp
Status: Succeeded

✗ RegisterModel step not found in pipeline


In [37]:
import boto3

sm = boto3.client('sagemaker', region_name='ap-southeast-1')

# List model packages in your registry group
response = sm.list_model_packages(
    ModelPackageGroupName='team03-CryptoScamDetector',
    SortOrder='Descending'
)

for pkg in response['ModelPackageSummaryList']:
    print(f"Model version: {pkg['ModelPackageVersion']}")
    print(f"Status: {pkg['ModelPackageStatus']}")
    print(f"Approval status: {pkg['ModelApprovalStatus']}")
    print()

Model version: 9
Status: Completed
Approval status: PendingManualApproval

Model version: 8
Status: Completed
Approval status: PendingManualApproval

Model version: 7
Status: Completed
Approval status: PendingManualApproval

Model version: 6
Status: Completed
Approval status: PendingManualApproval

Model version: 5
Status: Completed
Approval status: PendingManualApproval

Model version: 4
Status: Completed
Approval status: PendingManualApproval

Model version: 3
Status: Completed
Approval status: Approved

Model version: 2
Status: Completed
Approval status: Approved

Model version: 1
Status: Completed
Approval status: Approved



## 4. Optional: Continuous Monitoring Loop

`check_trigger_folder_once()` above is a single check — good for a
repeatable, gradeable demo. The cell below wraps it in a **bounded** loop to
show what continuous monitoring looks like in a live Jupyter kernel: it polls
every 30 seconds for up to 5 checks (2.5 minutes total), then stops. It is
intentionally capped so it cannot hang an automated notebook run — this is
the classroom stand-in for what an EventBridge rule watches for continuously
in production.

To see it actually catch something live, run this cell, then from a second
browser tab or terminal upload another CSV to
`s3://{BUCKET}/{TRIGGER_INCOMING_PREFIX}/` while it is running.

This cell is optional — skip it if you already ran section 3 above.

In [21]:
# POLL_INTERVAL_SECONDS = 30
# MAX_POLLS = 5

# for poll_number in range(1, MAX_POLLS + 1):
#     print(f"--- Poll {poll_number}/{MAX_POLLS} ---")
#     new_uris = check_trigger_folder_once(auto_start=True)
#     if not new_uris:
#         print("  (no new files)")
#     if poll_number < MAX_POLLS:
#         time.sleep(POLL_INTERVAL_SECONDS)

# print("\nMonitoring loop finished (reached MAX_POLLS). Re-run this cell to keep watching.")

## 5. How This Maps to a Production Setup

| Production component | This notebook's stand-in |
|---|---|
| S3 `PutObject` event | A new `.csv` appearing under `trigger/incoming/` |
| EventBridge rule (watches the S3 event) | The polling loop / `check_trigger_folder_once()` call in section 3-4 |
| Lambda function (calls `start_pipeline_execution`) | `pipeline_triggered.start(parameters={'InputDataUrl': ...})` inside `check_trigger_folder_once()` |
| CloudWatch Logs (execution history) | The notebook's own `print()` output, plus `pipeline_triggered.list_executions()` |
| IAM role restricting Lambda's S3/SageMaker access | The same team-scoped SageMaker execution role this notebook already runs under |

The tutor's material marks the full EventBridge -> Lambda -> Pipeline chain
(and GitHub Actions) as an **advanced extension**, skippable in a
multi-team classroom because of the extra IAM/secrets setup each team would
need. This simplified version demonstrates the same trigger *logic*
end-to-end without that setup.

## 6. Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `S3UploadFailedError: AccessDenied` when defining or starting the pipeline | The SageMaker SDK tried to upload something (estimator source, pipeline definition) outside the team's allowed S3 prefix | Confirm `pipeline_session = PipelineSession(default_bucket=BUCKET, default_bucket_prefix=PREFIX)` was used, not a bare `PipelineSession()` |
| `check_trigger_folder_once()` keeps re-triggering the same file | `trigger_seen_keys_team03.json` was deleted or never saved | Re-run the section 3 baseline cell, or check `SEEN_STATE_FILE.exists()` |
| `check_trigger_folder_once()` reports "No new files detected" even after uploading a test file | The exact same filename was uploaded before, so it's already in the seen-keys baseline | Use a new filename each time you re-test (e.g. change the `timestamp` variable in the synthetic-batch cell) |
| `PreprocessData` fails with `AlgorithmError` / `FileNotFoundError` | `preprocess.py` doesn't know the actual filename of whatever CSV triggered the run | Confirm `preprocess.py` discovers the input file by globbing `/opt/ml/processing/input/*.csv` rather than hardcoding a filename (fixed 1 Aug 2026) |
| `ClientError: ValidationException` on `pipeline_triggered.start()` | A parameter name/type mismatch (e.g. `InputDataUrl` passed as something other than a string S3 URI) | Print the URI before passing it; confirm it starts with `s3://` |
| `AUCQualityGate` always takes the `else` branch (nothing registers) | Test AUC genuinely below `QualityGateAUC` for this batch | Check `TrainModel`'s captured metrics (`FinalMetricDataList`) for that execution in Studio |
| New model versions not appearing in the Model Registry | `RegisterModel` only runs on the `if_steps` branch of `AUCQualityGate` -- it is skipped whenever the gate fails, by design | Check `step_condition`'s outcome for that execution in Studio, not just whether the pipeline "succeeded" |


In [36]:
print('=' * 55)
print('NOTEBOOK 05 COMPLETE')
print('=' * 55)
print(f'Triggered pipeline : {TRIGGERED_PIPELINE_NAME}')
print(f'Watch folder        : {TRIGGER_INCOMING_URI}')
print(f'Model Registry group: {MODEL_PACKAGE_GROUP} (shared with Notebook 03)')
print()
print('For Progress Check: run section 3 in Studio at least once so a real')
print('triggered execution exists that reaches the AUCQualityGate step and')
print('registers a model -- this is the "partial CI/CD pipeline setup" the')
print('rubric checks for at this stage.')
print('Full production automation (EventBridge/Lambda/GitHub Actions) is')
print('intentionally out of scope, per the tutor\'s own [SKIP THIS] marking.')


NOTEBOOK 05 COMPLETE
Triggered pipeline : iti113-team03-crypto-scam-detector-triggered
Watch folder        : s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/trigger/incoming
Model Registry group: team03-CryptoScamDetector (shared with Notebook 03)

For Progress Check: run section 3 in Studio at least once so a real
triggered execution exists that reaches the AUCQualityGate step and
registers a model -- this is the "partial CI/CD pipeline setup" the
rubric checks for at this stage.
Full production automation (EventBridge/Lambda/GitHub Actions) is
intentionally out of scope, per the tutor's own [SKIP THIS] marking.
